In [ ]:
import sys; sys.path.append('..')
import MeshFEM
import inflatables_parametrization as parametrization, sparse_matrices, mesh, numpy as np
from numpy.linalg import norm
import visualization

In [ ]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(1)
parallelism.set_gradient_assembly_num_threads(1)
parallelism.set_hessian_assembly_num_threads(1)

In [ ]:
m = mesh.Mesh("../../examples/lilium.msh")
# m = mesh.Mesh("../../examples/cone_test.obj")
lg = parametrization.LocalGlobalGenericParametrizer(m, parametrization.lscm(m))
# lg = parametrization.LocalGlobalParametrizer(m, parametrization.lscm(m))


In [ ]:
import tri_mesh_viewer
view = tri_mesh_viewer.TriMeshViewer(m)
view.show()

In [ ]:
for i in range(1000):
    lg.runIteration()
lg.energy()

In [ ]:
lg.alphaMin = 1.1
lg.alphaMax = 2.0

lg.betaMin = 1.1
lg.betaMax = 2.0
lg.energy()

In [ ]:
lg.runIteration()
lg.energy()

In [ ]:
import sys; sys.path.append('..');sys.path.append('../periodic_patches');

In [ ]:
augmented_x_scale_factors = np.load("../periodic_patches/Visualization/augmented_x_scale_factors.npy")
augmented_y_scale_factors = np.load("../periodic_patches/Visualization/augmented_y_scale_factors.npy")
augmented_pattern_parameters = np.load("../periodic_patches/Visualization/augmented_pattern_parameters.npy")
augmented_stiffness_coefficients = np.load("../periodic_patches/Visualization/augmented_stiffness_coefficients.npy")

In [ ]:
import parametrization_helper, importlib

In [ ]:
importlib.reload(parametrization_helper)

In [ ]:
lines = np.array([[-1.        , -1.        ,  2.415     ],
       [-2.30769231, -1.        ,  3.81923077],
       [ 0.77304965,  1.        , -2.27304965],
       [ 1.29357798,  1.        , -2.94036697],
       [-0.43333333, -1.        ,  1.655     ]])

In [ ]:
splines = parametrization_helper.get_stiffness_coefficient_splines(augmented_x_scale_factors, augmented_y_scale_factors, augmented_stiffness_coefficients, lines)

In [ ]:
splines[0](1.1, 1.1)

In [ ]:
# rparam = parametrization.RegularizedGenericParametrizer(lg, splines)
# # rparam.useBarrier = True
# # rparam.barrierA = 0.3
# # rparam.barrierB = 100
# rparam.setLines(lines)

In [ ]:
rsvd = parametrization.RegularizedGenericParametrizer(lg, splines)
# rsvd = parametrization.RegularizedParametrizerSVD(m, lg.uv())
rsvd.useBarrier = True
rsvd.useBarrier
rsvd.setLines(lines)


In [ ]:
rsvd.stretchRegW = 1.0
rsvd.stretchRegP = 2.0

In [ ]:
rsvd.diffRegW = 1e-6

In [ ]:
rsvd.phiRegW = 1.0
rsvd.phiRegP = 2.0

In [ ]:
rsvd.bendRegW = 10


In [ ]:
perturb = np.random.uniform(low=-1,high=1, size=rsvd.numVars())
fixedVars = np.arange(rsvd.phiOffset(), rsvd.numVars())
# fixedVars = []

In [ ]:
import fd_validation
# fd_validation.validateGrad(rsvd, fd_eps=1e-8,
#                            xeval=rsvd.getVars() + 1e-3 * np.random.uniform(low=-1, high=1, size=rsvd.numVars()),
#                            perturb=perturb[0:rsvd.numVars()], fixedVars=fixedVars)

In [ ]:
rsvd.energy(energyType = rsvd.EnergyType.BendingRegularization)

In [ ]:
rsvd.energy(energyType = rsvd.EnergyType.BendingRegularization)

In [ ]:
rsvd.gradient(energyType = rsvd.EnergyType.BendingRegularization)

In [ ]:
fd_validation.gradConvergencePlot(rsvd, customArgs = {"energyType": rsvd.EnergyType.Full})


In [ ]:
fd_validation.gradConvergencePlot(rsvd, customArgs = {"energyType": rsvd.EnergyType.Fitting})


In [ ]:
fd_validation.gradConvergencePlot(rsvd, customArgs = {"energyType": rsvd.EnergyType.PhiRegularization})

In [ ]:
fd_validation.gradConvergencePlot(rsvd, customArgs = {"energyType": rsvd.EnergyType.StretchRegularization})
# fd_validation.gradConvergencePlot(rsvd, customArgs = {"energyType": rsvd.EnergyType.DiffRegularization})

In [ ]:
fd_validation.gradConvergencePlot(rsvd, customArgs = {"energyType": rsvd.EnergyType.BendingRegularization})

In [ ]:
fd_validation.validateHessian(rsvd, fd_eps=1e-8)

In [ ]:
fd_validation.hessConvergencePlot(rsvd, customArgs={'projectionMask': False, "energyType": rsvd.EnergyType.Full})

In [ ]:
fd_validation.hessConvergencePlot(rsvd, customArgs={'projectionMask': False, "energyType": rsvd.EnergyType.Fitting})

In [ ]:
fd_validation.hessConvergencePlot(rsvd, customArgs={'projectionMask': False, "energyType": rsvd.EnergyType.BendingRegularization})

In [ ]:
fd_validation.hessConvergencePlot(rsvd, customArgs={'projectionMask': True, "energyType": rsvd.EnergyType.StretchRegularization})

In [ ]:
fd_validation.hessConvergencePlot(rsvd, customArgs={'projectionMask': True, "energyType": rsvd.EnergyType.PhiRegularization})

Errors in the Hessian for $1< p_\phi < 2$ happen due to discarded Hessian contributions near the singularity at $\sin(\phi_i - \phi_j) = 0$. This is validated by the following code, which verifies that the worst error in the predicted gradient change originates from a pair of triangles with $|\sin(\phi_i - \phi_j)|$ below the discard threshold. Note that the majority of the Hessian information is still good.

In [ ]:
rsvd.phiRegW = 1
rsvd.phiRegP = 1.5

In [ ]:
xeval = rsvd.getVars() + 1e-3 * np.random.uniform(low=-1, high=1, size=rsvd.numVars())
perturb2 = np.random.uniform(low=-1,high=1, size=rsvd.numVars())

In [ ]:
H_err = fd_validation.validateHessian(rsvd, fd_eps=1e-9,
                                      xeval=xeval, perturb=perturb2)
fd_delta_grad, an_delta_grad = H_err[1:]
H_err

In [ ]:
worst = np.argmax(np.abs(an_delta_grad - fd_delta_grad))
(an_delta_grad[worst-2:worst+2], fd_delta_grad[worst-2:worst+2])

In [ ]:
rsvd.setVars(xeval)
phis = rsvd.leftStretchAngles()

In [ ]:
result = []
worst_vtx = worst if worst < rsvd.vOffset() else worst - rsvd.vOffset()
for i in np.where(np.array(m.triangles()) == worst_vtx)[0]:
    result.append([(i, j, phis[i], phis[j], np.abs(np.sin(phis[i] - phis[j]))) for j in m.trisAdjTri(i)])
result